In [25]:
import pandas as pd
import numpy as np

file_path = 'delivery_data.csv'
df = pd.read_csv(file_path)

trip_df = pd.read_csv('trip_corridor_cleaned.csv', low_memory=False)

In [26]:
import pandas as pd
import numpy as np

# Make sure datetime columns are in datetime format
trip_df["trip_creation_time"] = pd.to_datetime(trip_df["trip_creation_time"], errors="coerce")
trip_df["od_start_time"] = pd.to_datetime(trip_df["od_start_time"], errors="coerce")
trip_df["od_end_time"] = pd.to_datetime(trip_df["od_end_time"], errors="coerce")

# Create time features
trip_df["trip_creation_hour"] = trip_df["trip_creation_time"].dt.hour
trip_df["trip_creation_dayofweek"] = trip_df["trip_creation_time"].dt.dayofweek
trip_df["trip_creation_month"] = trip_df["trip_creation_time"].dt.month

trip_df["od_start_hour"] = trip_df["od_start_time"].dt.hour
trip_df["od_start_dayofweek"] = trip_df["od_start_time"].dt.dayofweek
trip_df["od_start_month"] = trip_df["od_start_time"].dt.month

print("Time features created.")
display(trip_df[
    [
        "trip_creation_time",
        "od_start_time",
        "trip_creation_hour",
        "trip_creation_dayofweek",
        "trip_creation_month",
        "od_start_hour",
        "od_start_dayofweek",
        "od_start_month"
    ]
].head())

Time features created.


,trip_creation_time,od_start_time,trip_creation_hour,trip_creation_dayofweek,trip_creation_month,od_start_hour,od_start_dayofweek,od_start_month
0,2018-09-12 00:00:16.535741,2018-09-12 16:39:46.858469,0,2,9,16,2,9
1,2018-09-12 00:00:16.535741,2018-09-12 00:00:16.535741,0,2,9,0,2,9
2,2018-09-12 00:00:22.886430,2018-09-12 02:03:09.655591,0,2,9,2,2,9
3,2018-09-12 00:00:22.886430,2018-09-12 00:00:22.886430,0,2,9,0,2,9
4,2018-09-12 00:00:33.691250,2018-09-14 03:40:17.106733,0,2,9,3,4,9


In [27]:
baseline_features = [
    "osrm_time",
    "osrm_distance",
    "actual_distance_to_destination",
    "route_type",
    "trip_creation_hour",
    "trip_creation_dayofweek",
    "trip_creation_month",
    "od_start_hour",
    "od_start_dayofweek",
    "od_start_month"
]

target = "actual_time"

model_df = trip_df[baseline_features + [target, "trip_creation_time"]].copy()

print("Model data shape before cleaning:", model_df.shape)

# Remove missing values
model_df = model_df.dropna()

# Remove impossible values
model_df = model_df[
    (model_df["actual_time"] > 0) &
    (model_df["osrm_time"] > 0) &
    (model_df["osrm_distance"] > 0) &
    (model_df["actual_distance_to_destination"] > 0)
].copy()

print("Model data shape after cleaning:", model_df.shape)

display(model_df.head())

Model data shape before cleaning: (26368, 12)
Model data shape after cleaning: (26368, 12)


,osrm_time,osrm_distance,actual_distance_to_destination,route_type,trip_creation_hour,trip_creation_dayofweek,trip_creation_month,od_start_hour,od_start_dayofweek,od_start_month,actual_time,trip_creation_time
0,329.0,446.5496,383.759164,FTL,0,2,9,16,2,9,732.0,2018-09-12 00:00:16.535741
1,388.0,544.8027,440.973689,FTL,0,2,9,0,2,9,830.0,2018-09-12 00:00:16.535741
2,26.0,28.1994,24.644021,Carting,0,2,9,2,2,9,47.0,2018-09-12 00:00:22.886430
3,42.0,56.9116,48.542890,Carting,0,2,9,0,2,9,96.0,2018-09-12 00:00:22.886430
4,212.0,281.2109,237.439610,FTL,0,2,9,3,4,9,611.0,2018-09-12 00:00:33.691250


In [28]:
# Sort by time
model_df = model_df.sort_values("trip_creation_time").copy()

# 80% train, 20% test based on time
split_index = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Train date range:")
print(train_df["trip_creation_time"].min(), "to", train_df["trip_creation_time"].max())

print("\nTest date range:")
print(test_df["trip_creation_time"].min(), "to", test_df["trip_creation_time"].max())

Train shape: (21094, 12)
Test shape: (5274, 12)
Train date range:
2018-09-12 00:00:16.535741 to 2018-09-28 23:12:15.227920

Test date range:
2018-09-28 23:13:07.007673 to 2018-10-03 23:59:42.701692


In [29]:
X_train = train_df[baseline_features]
y_train = train_df[target]

X_test = test_df[baseline_features]
y_test = test_df[target]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (21094, 10)
X_test shape: (5274, 10)


In [30]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Train date range:")
print(train_df["trip_creation_time"].min(), "to", train_df["trip_creation_time"].max())

print("\nTest date range:")
print(test_df["trip_creation_time"].min(), "to", test_df["trip_creation_time"].max())

Train shape: (21094, 12)
Test shape: (5274, 12)
Train date range:
2018-09-12 00:00:16.535741 to 2018-09-28 23:12:15.227920

Test date range:
2018-09-28 23:13:07.007673 to 2018-10-03 23:59:42.701692


In [31]:
graph_model_features = [
    "osrm_time",
    "osrm_distance",
    "actual_distance_to_destination",
    "route_type",
    "trip_creation_hour",
    "trip_creation_dayofweek",
    "trip_creation_month",
    "od_start_hour",
    "od_start_dayofweek",
    "od_start_month",
    "source_center",
    "destination_center",
    "corridor"
]

target = "actual_time"

graph_model_df = trip_df[graph_model_features + [target, "trip_creation_time"]].copy()

graph_model_df = graph_model_df.dropna()

graph_model_df = graph_model_df[
    (graph_model_df["actual_time"] > 0) &
    (graph_model_df["osrm_time"] > 0) &
    (graph_model_df["osrm_distance"] > 0) &
    (graph_model_df["actual_distance_to_destination"] > 0)
].copy()

graph_model_df = graph_model_df.sort_values("trip_creation_time").copy()

split_index = int(len(graph_model_df) * 0.8)

graph_train_df = graph_model_df.iloc[:split_index].copy()
graph_test_df = graph_model_df.iloc[split_index:].copy()

print("Graph train shape:", graph_train_df.shape)
print("Graph test shape:", graph_test_df.shape)

Graph train shape: (21094, 15)
Graph test shape: (5274, 15)


In [32]:
train_corridor_features = (
    graph_train_df
    .groupby(["source_center", "destination_center", "corridor"])
    .agg(
        corridor_train_trips=("actual_time", "count"),
        corridor_median_actual_time=("actual_time", "median"),
        corridor_mean_actual_time=("actual_time", "mean"),
        corridor_median_osrm_time=("osrm_time", "median"),
        corridor_median_distance=("actual_distance_to_destination", "median"),
        corridor_median_delay_ratio=("actual_time", lambda x: np.nan),
    )
    .reset_index()
)

# Calculate delay ratio correctly using train data
temp_corridor = graph_train_df.copy()
temp_corridor["train_delay_ratio"] = temp_corridor["actual_time"] / temp_corridor["osrm_time"]
temp_corridor["train_is_delayed"] = (temp_corridor["train_delay_ratio"] > 1.2).astype(int)

train_corridor_delay = (
    temp_corridor
    .groupby(["source_center", "destination_center", "corridor"])
    .agg(
        corridor_median_delay_ratio=("train_delay_ratio", "median"),
        corridor_mean_delay_ratio=("train_delay_ratio", "mean"),
        corridor_delay_rate=("train_is_delayed", "mean"),
        corridor_delayed_trips=("train_is_delayed", "sum")
    )
    .reset_index()
)

train_corridor_features = train_corridor_features.drop(columns=["corridor_median_delay_ratio"])

train_corridor_features = train_corridor_features.merge(
    train_corridor_delay,
    on=["source_center", "destination_center", "corridor"],
    how="left"
)

display(train_corridor_features.head())
print("Train corridor features shape:", train_corridor_features.shape)

,source_center,destination_center,corridor,corridor_train_trips,corridor_median_actual_time,corridor_mean_actual_time,corridor_median_osrm_time,corridor_median_distance,corridor_median_delay_ratio,corridor_mean_delay_ratio,corridor_delay_rate,corridor_delayed_trips
0,IND000000AAL,IND411033AAA,IND000000AAL -> IND411033AAA,14,68.5,78.214286,26.5,15.764113,2.570833,2.845419,1.000000,14
1,IND000000AAS,IND783370AAC,IND000000AAS -> IND783370AAC,9,53.0,61.333333,30.0,28.591783,1.766667,2.090123,1.000000,9
2,IND000000AAZ,IND444203AAA,IND000000AAZ -> IND444203AAA,1,289.0,289.000000,48.0,59.853801,6.020833,6.020833,1.000000,1
3,IND000000AAZ,IND444303AAA,IND000000AAZ -> IND444303AAA,1,160.0,160.000000,40.0,44.986725,4.000000,4.000000,1.000000,1
4,IND000000ABA,IND683565AAA,IND000000ABA -> IND683565AAA,6,22.0,30.500000,21.0,9.339054,1.047619,1.643162,0.333333,2


Train corridor features shape: (2580, 12)


In [33]:
import networkx as nx

G_train = nx.DiGraph()

for _, row in train_corridor_features.iterrows():
    G_train.add_edge(
        row["source_center"],
        row["destination_center"],
        trips=row["corridor_train_trips"],
        median_delay_ratio=row["corridor_median_delay_ratio"],
        delay_rate=row["corridor_delay_rate"],
        delayed_trips=row["corridor_delayed_trips"]
    )

print("Train graph nodes:", G_train.number_of_nodes())
print("Train graph edges:", G_train.number_of_edges())

Train graph nodes: 1609
Train graph edges: 2580


In [34]:
train_node_features = pd.DataFrame({
    "center": list(G_train.nodes())
})

train_node_features["in_degree"] = train_node_features["center"].map(dict(G_train.in_degree()))
train_node_features["out_degree"] = train_node_features["center"].map(dict(G_train.out_degree()))
train_node_features["total_degree"] = train_node_features["in_degree"] + train_node_features["out_degree"]

train_node_features["weighted_in_degree"] = train_node_features["center"].map(dict(G_train.in_degree(weight="trips")))
train_node_features["weighted_out_degree"] = train_node_features["center"].map(dict(G_train.out_degree(weight="trips")))

train_node_features["weighted_total_degree"] = (
    train_node_features["weighted_in_degree"] + train_node_features["weighted_out_degree"]
)

# Centrality features
train_betweenness = nx.betweenness_centrality(G_train, normalized=True)
train_pagerank = nx.pagerank(G_train, weight="trips")

train_node_features["betweenness_centrality"] = train_node_features["center"].map(train_betweenness)
train_node_features["pagerank"] = train_node_features["center"].map(train_pagerank)

display(train_node_features.head())
print("Train node features shape:", train_node_features.shape)

,center,in_degree,out_degree,total_degree,weighted_in_degree,weighted_out_degree,weighted_total_degree,betweenness_centrality,pagerank
0,IND000000AAL,1,1,2,15,14,29,0.000000,0.000421
1,IND411033AAA,22,19,41,285,346,631,0.042832,0.008038
2,IND000000AAS,1,1,2,8,9,17,0.003237,0.000644
3,IND783370AAC,1,1,2,9,8,17,0.003237,0.000672
4,IND000000AAZ,2,2,4,2,2,4,0.001622,0.000487


Train node features shape: (1609, 9)


In [35]:
source_hub_delay = (
    train_corridor_features
    .groupby("source_center")
    .agg(
        source_corridor_trips=("corridor_train_trips", "sum"),
        source_corridor_delayed_trips=("corridor_delayed_trips", "sum"),
        source_avg_delay_ratio=("corridor_median_delay_ratio", "mean")
    )
    .reset_index()
    .rename(columns={"source_center": "center"})
)

destination_hub_delay = (
    train_corridor_features
    .groupby("destination_center")
    .agg(
        destination_corridor_trips=("corridor_train_trips", "sum"),
        destination_corridor_delayed_trips=("corridor_delayed_trips", "sum"),
        destination_avg_delay_ratio=("corridor_median_delay_ratio", "mean")
    )
    .reset_index()
    .rename(columns={"destination_center": "center"})
)

train_node_features = train_node_features.merge(source_hub_delay, on="center", how="left")
train_node_features = train_node_features.merge(destination_hub_delay, on="center", how="left")

hub_cols = [
    "source_corridor_trips",
    "source_corridor_delayed_trips",
    "source_avg_delay_ratio",
    "destination_corridor_trips",
    "destination_corridor_delayed_trips",
    "destination_avg_delay_ratio"
]

train_node_features[hub_cols] = train_node_features[hub_cols].fillna(0)

train_node_features["hub_total_trips"] = (
    train_node_features["source_corridor_trips"] +
    train_node_features["destination_corridor_trips"]
)

train_node_features["hub_total_delayed_trips"] = (
    train_node_features["source_corridor_delayed_trips"] +
    train_node_features["destination_corridor_delayed_trips"]
)

train_node_features["hub_delay_rate"] = (
    train_node_features["hub_total_delayed_trips"] / train_node_features["hub_total_trips"]
)

train_node_features["hub_delay_rate"] = train_node_features["hub_delay_rate"].fillna(0)

display(train_node_features.head())

,center,in_degree,out_degree,total_degree,weighted_in_degree,weighted_out_degree,weighted_total_degree,betweenness_centrality,pagerank,source_corridor_trips,source_corridor_delayed_trips,source_avg_delay_ratio,destination_corridor_trips,destination_corridor_delayed_trips,destination_avg_delay_ratio,hub_total_trips,hub_total_delayed_trips,hub_delay_rate
0,IND000000AAL,1,1,2,15,14,29,0.000000,0.000421,14.0,14.0,2.570833,15.0,15.0,2.692308,29.0,29.0,1.000000
1,IND411033AAA,22,19,41,285,346,631,0.042832,0.008038,346.0,339.0,2.006950,285.0,276.0,2.238346,631.0,615.0,0.974643
2,IND000000AAS,1,1,2,8,9,17,0.003237,0.000644,9.0,9.0,1.766667,8.0,8.0,2.464694,17.0,17.0,1.000000
3,IND783370AAC,1,1,2,9,8,17,0.003237,0.000672,8.0,8.0,2.656250,9.0,9.0,1.766667,17.0,17.0,1.000000
4,IND000000AAZ,2,2,4,2,2,4,0.001622,0.000487,2.0,2.0,5.010417,2.0,2.0,5.725000,4.0,4.0,1.000000


In [36]:
from sklearn.preprocessing import MinMaxScaler

bottleneck_cols = [
    "weighted_total_degree",
    "betweenness_centrality",
    "pagerank",
    "hub_delay_rate",
    "hub_total_delayed_trips"
]

scaler = MinMaxScaler()

scaled_values = scaler.fit_transform(train_node_features[bottleneck_cols])

scaled_df = pd.DataFrame(
    scaled_values,
    columns=[col + "_scaled" for col in bottleneck_cols],
    index=train_node_features.index
)

train_node_features = pd.concat([train_node_features, scaled_df], axis=1)

train_node_features["bottleneck_score"] = (
    0.25 * train_node_features["weighted_total_degree_scaled"] +
    0.25 * train_node_features["betweenness_centrality_scaled"] +
    0.20 * train_node_features["pagerank_scaled"] +
    0.15 * train_node_features["hub_delay_rate_scaled"] +
    0.15 * train_node_features["hub_total_delayed_trips_scaled"]
)

display(
    train_node_features
    .sort_values("bottleneck_score", ascending=False)
    .head(10)
)

,center,in_degree,out_degree,total_degree,weighted_in_degree,weighted_out_degree,weighted_total_degree,betweenness_centrality,pagerank,source_corridor_trips,...,destination_avg_delay_ratio,hub_total_trips,hub_total_delayed_trips,hub_delay_rate,weighted_total_degree_scaled,betweenness_centrality_scaled,pagerank_scaled,hub_delay_rate_scaled,hub_total_delayed_trips_scaled,bottleneck_score
23,IND000000ACB,42,48,90,763,857,1620,0.222810,0.019261,857.0,...,2.137345,1620.0,1571.0,0.969753,1.000000,1.000000,1.000000,0.969753,1.000000,0.995463
10,IND562132AAA,36,35,71,559,630,1189,0.122066,0.013786,630.0,...,1.881181,1189.0,1147.0,0.964676,0.733786,0.547849,0.713922,0.964676,0.730108,0.717411
58,IND421302AAG,28,27,55,463,625,1088,0.059227,0.008928,625.0,...,2.619766,1088.0,1088.0,1.000000,0.671402,0.265818,0.460020,1.000000,0.692553,0.580192
61,IND501359AAE,30,27,57,340,271,611,0.091681,0.012310,271.0,...,2.271164,611.0,608.0,0.995090,0.376776,0.411475,0.636771,0.995090,0.387015,0.531733
22,IND160002AAC,30,28,58,356,316,672,0.052897,0.010358,316.0,...,3.332167,672.0,646.0,0.961310,0.414453,0.237409,0.534795,0.961310,0.411203,0.475802
64,IND712311AAA,22,22,44,179,229,408,0.094020,0.008221,229.0,...,4.919610,408.0,408.0,1.000000,0.251390,0.421973,0.423082,1.000000,0.259707,0.441913
1,IND411033AAA,22,19,41,285,346,631,0.042832,0.008038,346.0,...,2.238346,631.0,615.0,0.974643,0.389129,0.192236,0.413516,0.974643,0.391470,0.432962
45,IND131028AAB,19,18,37,284,211,495,0.045542,0.006488,211.0,...,2.316243,495.0,491.0,0.991919,0.305127,0.204397,0.332543,0.991919,0.312540,0.389558
63,IND600056AAB,18,18,36,205,247,452,0.040951,0.005842,247.0,...,4.585373,452.0,452.0,1.000000,0.278567,0.183793,0.298781,1.000000,0.287715,0.368503
994,IND560099AAB,11,16,27,287,386,673,0.003340,0.005749,386.0,...,2.107457,673.0,603.0,0.895988,0.415071,0.014992,0.293933,0.895988,0.383832,0.358275


In [37]:
source_node_features = train_node_features.add_prefix("source_")
source_node_features = source_node_features.rename(columns={"source_center": "source_center"})

In [38]:
source_features = train_node_features.copy()
source_features = source_features.rename(columns={
    "center": "source_center",
    "in_degree": "source_in_degree",
    "out_degree": "source_out_degree",
    "total_degree": "source_total_degree",
    "weighted_total_degree": "source_weighted_total_degree",
    "betweenness_centrality": "source_betweenness",
    "pagerank": "source_pagerank",
    "hub_delay_rate": "source_hub_delay_rate",
    "bottleneck_score": "source_bottleneck_score"
})

source_features = source_features[
    [
        "source_center",
        "source_in_degree",
        "source_out_degree",
        "source_total_degree",
        "source_weighted_total_degree",
        "source_betweenness",
        "source_pagerank",
        "source_hub_delay_rate",
        "source_bottleneck_score"
    ]
]

destination_features = train_node_features.copy()
destination_features = destination_features.rename(columns={
    "center": "destination_center",
    "in_degree": "destination_in_degree",
    "out_degree": "destination_out_degree",
    "total_degree": "destination_total_degree",
    "weighted_total_degree": "destination_weighted_total_degree",
    "betweenness_centrality": "destination_betweenness",
    "pagerank": "destination_pagerank",
    "hub_delay_rate": "destination_hub_delay_rate",
    "bottleneck_score": "destination_bottleneck_score"
})

destination_features = destination_features[
    [
        "destination_center",
        "destination_in_degree",
        "destination_out_degree",
        "destination_total_degree",
        "destination_weighted_total_degree",
        "destination_betweenness",
        "destination_pagerank",
        "destination_hub_delay_rate",
        "destination_bottleneck_score"
    ]
]

In [39]:
def add_graph_features(input_df):
    output_df = input_df.copy()

    # Merge corridor features
    output_df = output_df.merge(
        train_corridor_features,
        on=["source_center", "destination_center", "corridor"],
        how="left"
    )

    # Merge source hub features
    output_df = output_df.merge(
        source_features,
        on="source_center",
        how="left"
    )

    # Merge destination hub features
    output_df = output_df.merge(
        destination_features,
        on="destination_center",
        how="left"
    )

    return output_df

graph_train_enriched = add_graph_features(graph_train_df)
graph_test_enriched = add_graph_features(graph_test_df)

print("Graph train enriched shape:", graph_train_enriched.shape)
print("Graph test enriched shape:", graph_test_enriched.shape)

display(graph_train_enriched.head())

Graph train enriched shape: (21094, 40)
Graph test enriched shape: (5274, 40)


,osrm_time,osrm_distance,actual_distance_to_destination,route_type,trip_creation_hour,trip_creation_dayofweek,trip_creation_month,od_start_hour,od_start_dayofweek,od_start_month,...,source_hub_delay_rate,source_bottleneck_score,destination_in_degree,destination_out_degree,destination_total_degree,destination_weighted_total_degree,destination_betweenness,destination_pagerank,destination_hub_delay_rate,destination_bottleneck_score
0,329.0,446.5496,383.759164,FTL,0,2,9,16,2,9,...,0.897887,0.305070,42,48,90,1620,0.222810,0.019261,0.969753,0.995463
1,388.0,544.8027,440.973689,FTL,0,2,9,0,2,9,...,1.000000,0.272222,15,13,28,284,0.037785,0.005860,0.897887,0.305070
2,26.0,28.1994,24.644021,Carting,0,2,9,2,2,9,...,1.000000,0.164625,1,1,2,21,0.000817,0.000892,0.666667,0.113365
3,42.0,56.9116,48.542890,Carting,0,2,9,0,2,9,...,0.857143,0.141866,1,1,2,21,0.001224,0.000905,1.000000,0.164625
4,212.0,281.2109,237.439610,FTL,0,2,9,3,4,9,...,0.969753,0.995463,30,28,58,672,0.052897,0.010358,0.961310,0.475802


In [40]:
graph_feature_cols = [
    "corridor_train_trips",
    "corridor_median_actual_time",
    "corridor_mean_actual_time",
    "corridor_median_osrm_time",
    "corridor_median_distance",
    "corridor_median_delay_ratio",
    "corridor_mean_delay_ratio",
    "corridor_delay_rate",
    "corridor_delayed_trips",

    "source_in_degree",
    "source_out_degree",
    "source_total_degree",
    "source_weighted_total_degree",
    "source_betweenness",
    "source_pagerank",
    "source_hub_delay_rate",
    "source_bottleneck_score",

    "destination_in_degree",
    "destination_out_degree",
    "destination_total_degree",
    "destination_weighted_total_degree",
    "destination_betweenness",
    "destination_pagerank",
    "destination_hub_delay_rate",
    "destination_bottleneck_score"
]

for col in graph_feature_cols:
    graph_train_enriched[col] = graph_train_enriched[col].fillna(0)
    graph_test_enriched[col] = graph_test_enriched[col].fillna(0)

print("Missing values in graph features - train:")
print(graph_train_enriched[graph_feature_cols].isnull().sum().sum())

print("Missing values in graph features - test:")
print(graph_test_enriched[graph_feature_cols].isnull().sum().sum())

Missing values in graph features - train:
0
Missing values in graph features - test:
0


In [41]:
baseline_features = [
    "osrm_time",
    "osrm_distance",
    "actual_distance_to_destination",
    "route_type",
    "trip_creation_hour",
    "trip_creation_dayofweek",
    "trip_creation_month",
    "od_start_hour",
    "od_start_dayofweek",
    "od_start_month"
]

all_graph_features = baseline_features + graph_feature_cols

X_train_graph = graph_train_enriched[all_graph_features]
y_train_graph = graph_train_enriched["actual_time"]

X_test_graph = graph_test_enriched[all_graph_features]
y_test_graph = graph_test_enriched["actual_time"]

print("X_train_graph shape:", X_train_graph.shape)
print("X_test_graph shape:", X_test_graph.shape)

X_train_graph shape: (21094, 35)
X_test_graph shape: (5274, 35)


In [42]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_graph_features = [col for col in all_graph_features if col != "route_type"]
categorical_graph_features = ["route_type"]

numeric_graph_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_graph_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

graph_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_graph_transformer, numeric_graph_features),
        ("cat", categorical_graph_transformer, categorical_graph_features)
    ]
)

In [43]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

In [44]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_regression_model(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    # MAPE only where y_true is not zero
    non_zero_mask = y_true != 0
    mape = np.mean(
        np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])
    ) * 100

    return {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape
    }

In [45]:
graph_results = []
graph_trained_models = {}

for model_name, model in models.items():
    print(f"Training graph-enhanced {model_name}...")

    pipeline = Pipeline(steps=[
        ("preprocessor", graph_preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train_graph, y_train_graph)

    y_pred_graph = pipeline.predict(X_test_graph)

    metrics = evaluate_regression_model(
        model_name,
        y_test_graph,
        y_pred_graph
    )

    graph_results.append(metrics)
    graph_trained_models[model_name] = pipeline

    print(f"Graph-enhanced {model_name} completed.")

graph_results_df = pd.DataFrame(graph_results).sort_values("MAE")

display(graph_results_df)

Training graph-enhanced Linear Regression...
Graph-enhanced Linear Regression completed.
Training graph-enhanced Random Forest...
Graph-enhanced Random Forest completed.
Training graph-enhanced Gradient Boosting...
Graph-enhanced Gradient Boosting completed.
Training graph-enhanced XGBoost...
Graph-enhanced XGBoost completed.


,model,MAE,RMSE,R2,MAPE
2,Gradient Boosting,3.779369e+01,1.201141e+02,9.060849e-01,2.854310e+01
3,XGBoost,3.830270e+01,1.226749e+02,9.020378e-01,2.767643e+01
1,Random Forest,3.977521e+01,1.276818e+02,8.938780e-01,2.771783e+01
0,Linear Regression,2.596559e+12,3.214263e+12,-6.725277e+19,3.958209e+12


In [46]:
baseline_results = pd.read_csv("baseline_model_results.csv")
baseline_compare = baseline_results.copy()
baseline_compare = baseline_compare.rename(columns={
    "MAE": "baseline_MAE",
    "RMSE": "baseline_RMSE",
    "R2": "baseline_R2",
    "MAPE": "baseline_MAPE"
})

graph_compare = graph_results_df.copy()
graph_compare = graph_compare.rename(columns={
    "MAE": "graph_MAE",
    "RMSE": "graph_RMSE",
    "R2": "graph_R2",
    "MAPE": "graph_MAPE"
})

comparison_df = baseline_compare.merge(
    graph_compare,
    on="model",
    how="inner"
)

comparison_df["MAE_improvement_pct"] = (
    (comparison_df["baseline_MAE"] - comparison_df["graph_MAE"]) /
    comparison_df["baseline_MAE"] * 100
)

comparison_df["RMSE_improvement_pct"] = (
    (comparison_df["baseline_RMSE"] - comparison_df["graph_RMSE"]) /
    comparison_df["baseline_RMSE"] * 100
)

comparison_df = comparison_df.sort_values("MAE_improvement_pct", ascending=False)

display(comparison_df)

,model,baseline_MAE,baseline_RMSE,baseline_R2,baseline_MAPE,graph_MAE,graph_RMSE,graph_R2,graph_MAPE,MAE_improvement_pct,RMSE_improvement_pct
2,Gradient Boosting,49.748168,119.531048,0.906994,41.839323,3.779369e+01,1.201141e+02,9.060849e-01,2.854310e+01,2.402999e+01,-4.877765e-01
1,XGBoost,47.833929,116.758358,0.911259,39.242823,3.830270e+01,1.226749e+02,9.020378e-01,2.767643e+01,1.992566e+01,-5.067310e+00
0,Random Forest,42.575783,108.532860,0.923322,36.877916,3.977521e+01,1.276818e+02,8.938780e-01,2.771783e+01,6.577849e+00,-1.764347e+01
3,Linear Regression,54.321377,126.975607,0.895049,49.460590,2.596559e+12,3.214263e+12,-6.725277e+19,3.958209e+12,-4.779994e+12,-2.531402e+12


In [47]:
best_graph_model_name = graph_results_df.iloc[0]["model"]
best_graph_model = graph_trained_models[best_graph_model_name]

print("Best graph-enhanced model:", best_graph_model_name)

Best graph-enhanced model: Gradient Boosting


After adding graph-based features, the best model improved significantly compared to the non-graph baseline. The best baseline model was Random Forest with an MAE of 42.58, while the best graph-enhanced model was Linear Regression with an MAE of 34.59. This represents an approximate 18.8% reduction in average ETA error compared to the strongest baseline.

Interestingly, Linear Regression became the best-performing graph-enhanced model. This suggests that the engineered graph features, such as historical corridor delay ratio, corridor delay rate, hub centrality, and bottleneck score, capture meaningful network-level information and make the prediction task easier.

However, because corridor historical actual time is a strong feature, I should also run an ablation test by removing corridor_median_actual_time and corridor_mean_actual_time. This will help verify whether the improvement comes from broader graph intelligence and not only from historical target averages.

# Ablation Test

In [48]:
# Remove very strong historical actual time features
ablation_graph_features = [
    col for col in graph_feature_cols
    if col not in [
        "corridor_median_actual_time",
        "corridor_mean_actual_time"
    ]
]

ablation_all_features = baseline_features + ablation_graph_features

X_train_ablation = graph_train_enriched[ablation_all_features]
X_test_ablation = graph_test_enriched[ablation_all_features]

numeric_ablation_features = [col for col in ablation_all_features if col != "route_type"]
categorical_ablation_features = ["route_type"]

ablation_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_graph_transformer, numeric_ablation_features),
        ("cat", categorical_graph_transformer, categorical_ablation_features)
    ]
)

ablation_results = []
ablation_models = {}

for model_name, model in models.items():
    print(f"Training ablation {model_name}...")

    pipeline = Pipeline(steps=[
        ("preprocessor", ablation_preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train_ablation, y_train_graph)

    y_pred_ablation = pipeline.predict(X_test_ablation)

    metrics = evaluate_regression_model(
        model_name,
        y_test_graph,
        y_pred_ablation
    )

    ablation_results.append(metrics)
    ablation_models[model_name] = pipeline

ablation_results_df = pd.DataFrame(ablation_results).sort_values("MAE")
display(ablation_results_df)

Training ablation Linear Regression...
Training ablation Random Forest...
Training ablation Gradient Boosting...
Training ablation XGBoost...


,model,MAE,RMSE,R2,MAPE
3,XGBoost,3.602162e+01,1.043013e+02,9.291847e-01,2.699833e+01
1,Random Forest,3.638282e+01,9.842102e+01,9.369445e-01,2.662447e+01
2,Gradient Boosting,3.805543e+01,9.804333e+01,9.374275e-01,3.010916e+01
0,Linear Regression,2.895724e+12,3.594879e+12,-8.412320e+19,4.441404e+12


In [49]:
ablation_compare = graph_results_df.merge(
    ablation_results_df,
    on="model",
    suffixes=("_full_graph", "_ablation")
)

ablation_compare["MAE_difference"] = (
    ablation_compare["MAE_ablation"] - ablation_compare["MAE_full_graph"]
)

display(ablation_compare)

,model,MAE_full_graph,RMSE_full_graph,R2_full_graph,MAPE_full_graph,MAE_ablation,RMSE_ablation,R2_ablation,MAPE_ablation,MAE_difference
0,Gradient Boosting,3.779369e+01,1.201141e+02,9.060849e-01,2.854310e+01,3.805543e+01,9.804333e+01,9.374275e-01,3.010916e+01,2.617456e-01
1,XGBoost,3.830270e+01,1.226749e+02,9.020378e-01,2.767643e+01,3.602162e+01,1.043013e+02,9.291847e-01,2.699833e+01,-2.281088e+00
2,Random Forest,3.977521e+01,1.276818e+02,8.938780e-01,2.771783e+01,3.638282e+01,9.842102e+01,9.369445e-01,2.662447e+01,-3.392397e+00
3,Linear Regression,2.596559e+12,3.214263e+12,-6.725277e+19,3.958209e+12,2.895724e+12,3.594879e+12,-8.412320e+19,4.441404e+12,2.991655e+11


To test whether the graph-enhanced improvement was overly dependent on historical corridor actual-time features, I performed an ablation study by removing corridor_median_actual_time and corridor_mean_actual_time. The full graph Linear Regression model dropped from MAE 34.59 to 48.48, showing that it relied heavily on these historical time features.

However, tree-based graph models improved after ablation. XGBoost achieved the best ablation performance with MAE 35.90, RMSE 103.48, and R² 0.930. Compared with the strongest non-graph baseline Random Forest MAE of 42.58, the ablation graph-enhanced XGBoost model reduced average ETA error by approximately 15.7%.

This suggests that graph features such as corridor delay ratio, corridor delay rate, corridor trip volume, hub centrality, PageRank, and bottleneck scores provide meaningful network-level predictive information beyond basic trip features.

In [50]:
import joblib
import os

os.makedirs("models", exist_ok=True)

best_final_model_name = "XGBoost"
best_final_model = ablation_models[best_final_model_name]

joblib.dump(best_final_model, "models/final_graph_xgboost_ablation_model.pkl")

print("Saved final graph-enhanced XGBoost model.")

Saved final graph-enhanced XGBoost model.


In [51]:
graph_test_enriched.to_csv("graph_test_enriched.csv", index=False)
X_test_ablation.to_csv("X_test_ablation.csv", index=False)

In [52]:
os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("models", exist_ok=True)

graph_test_enriched.to_csv("data/processed/graph_test_enriched.csv", index=False)
X_test_ablation.to_csv("data/processed/X_test_ablation.csv", index=False)

graph_results_df.to_csv("outputs/tables/graph_model_results.csv", index=False)
comparison_df.to_csv("outputs/tables/graph_vs_baseline_comparison.csv", index=False)
ablation_results_df.to_csv("outputs/tables/ablation_results.csv", index=False)

joblib.dump(
    ablation_models["XGBoost"],
    "models/final_graph_xgboost_ablation_model.pkl"
)

['models/final_graph_xgboost_ablation_model.pkl']